# Coding Automation Agent — Notebook Chat Interface

This notebook provides a simple chat UI to send text to the `CodingAutomationAgent` located at `src/agents/coding_automation_agent.py`.

Usage: run the cells in order to start the chat widget. Use the following commands in the input area:
- `scaffold <site_name>` — create a basic website scaffold under `src/agents/generated/<site_name>`.
- `generate filename: <filename>\n<code>` — create a file with the provided filename and contents.
- `run <shell command>` — run a shell command (be careful).
- Any other text will be echoed with suggestions.


In [1]:
# Install missing packages
%pip install ipywidgets
%pip install nbformat
%pip install nbclient

# Import required libraries
import os
import sys
from pathlib import Path
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
# tools for programmatic notebook manipulation
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Safely load the agent module by file path (avoid importing package top-level code)
from importlib import util
from pathlib import Path

# from examples.tools.image_generator import open_file

# Define the agent_file variable
agent_file = Path('src/agents/coding_automation_agent.py')

def new_func():
    # Create a minimal placeholder agent file if it does not exist
    agent_file.parent.mkdir(parents=True, exist_ok=True)
    agent_file.write_text(
        "class CodingAutomationAgent:\n"
        "    def __init__(self):\n"
        "        pass\n"
        "    def scaffold_website(self, site_name):\n"
        "        return f'Scaffolded {site_name}'\n"
        "    def generate_code(self, code, filename):\n"
        "        return filename\n"
        "    def automate_task(self, cmd):\n"
        "        return 0\n"
    )
    print(f'Created placeholder agent file: {agent_file}')

if not agent_file.exists():
    new_func()

spec = util.spec_from_file_location('coding_automation_agent', str(agent_file))
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load spec for {agent_file}")
agent_mod = util.module_from_spec(spec)
# execute module code
spec.loader.exec_module(agent_mod)
# grab the class and instantiate
if not hasattr(agent_mod, 'CodingAutomationAgent'):
    raise AttributeError(f"'CodingAutomationAgent' class not found in {agent_file}")
CodingAutomationAgent = getattr(agent_mod, 'CodingAutomationAgent')
agent = CodingAutomationAgent()
agent


In [3]:
def handle_input(text: str) -> str:
    text = text.strip()
    if not text:
        return 'No input provided.'
    # scaffold command
    if text.startswith('scaffold '):
        _, site_name = text.split(' ', 1)
        try:
            path = agent.scaffold_website(site_name.strip())
            return f'Scaffolded site at: {path}'
        except Exception as e:
            return f'Error scaffolding site: {e}'
    # generate command: detect 'generate filename: <filename>\n<code>'
    if text.startswith('generate filename:'):
        parts = text.split('\n', 1)
        header = parts[0]
        code = parts[1] if len(parts) > 1 else ''
        _, fname = header.split(':', 1)
        filename = Path(fname.strip()).name  # sanitize filename
        try:
            # write into the agent's BASE_DIR by not passing a custom directory
            filepath = agent.generate_code(code, filename)
            return f'Generated file: {filepath}'
        except Exception as e:
            return f'Error generating file: {e}'
    # run shell command
    if text.startswith('run '):
        _, cmd = text.split(' ', 1)
        rc = agent.automate_task(cmd)
        return f'Command exited with status {rc}'
    # default behavior: echo and suggestions
    return ("Echo: " + text + '\n\n'
            "Try commands like:\n"
            "- scaffold mysite\n"
            "- generate filename: hello.py\\nprint(\\'hi\\')\n")


In [8]:
# Build chat UI with file preview/download links
input_area = widgets.Textarea(placeholder='Enter command or message here...', layout=widgets.Layout(width='100%', height='120px'))
send_btn = widgets.Button(description='Send', button_style='primary')
clear_btn = widgets.Button(description='Clear', button_style='warning')
output = widgets.Output(layout={'border': '1px solid #ddd'})

from IPython.display import FileLink, Markdown


def _display_file_preview(path_str: str):
    """Display a clickable file link and a small code preview if the file is text."""
    try:
        p = Path(path_str)
        if not p.exists():
            display(Markdown(f'**File not found:** `{path_str}`'))
            return
        # Show clickable link
        display(Markdown(f'- [Open file]({p.as_posix()})'))
        try:
            # Read small files only (limit to 100KB)
            size = p.stat().st_size
            if size < 100 * 1024 and p.suffix in ('.py', '.txt', '.md', '.html', '.css', '.js'):
                text = p.read_text(encoding='utf-8')
                # show as a fenced code block with language guessing from suffix
                lang = p.suffix.lstrip('.') or 'text'
                display(Markdown(f'```{lang}\n{text}\n```'))
            else:
                display(Markdown(f'**File is large or binary (size={size} bytes); download via link above.**'))
        except Exception as e:
            display(Markdown(f'Could not read file for preview: {e}'))
    except Exception as e:
        display(Markdown(f'Error showing file link: {e}'))


def send_click(_):
    text = input_area.value
    with output:
        print('> You: ' + text)
        try:
            resp = handle_input(text)
            print('Agent:', resp)
            # If agent returned a generated file path, show a link & preview
            if isinstance(resp, str) and 'Generated file:' in resp:
                path = resp.split(':', 1)[1].strip()
                _display_file_preview(path)
            # If agent scaffolded a site, show index.html link and listing
            if isinstance(resp, str) and 'Scaffolded site at:' in resp:
                site_path = resp.split(':', 1)[1].strip()
                index_path = Path(site_path) / 'index.html'
                _display_file_preview(str(index_path))
                # show directory listing
                try:
                    files = sorted([str(p) for p in Path(site_path).iterdir()])
                    display(Markdown('**Site files:**'))
                    for f in files:
                        display(Markdown(f'- `{f}`'))
                except Exception as e:
                    display(Markdown(f'Could not list site files: {e}'))
        except Exception as e:
            print('Agent error:', repr(e))


def clear_click(_):
    output.clear_output()

send_btn.on_click(send_click)
clear_btn.on_click(clear_click)
ui = widgets.VBox([input_area, widgets.HBox([send_btn, clear_btn]), output])
display(ui)


In [7]:
# Examples to try in the chat input area:
print('Examples:\n- scaffold demo_site\n- generate filename: example.py\n  print(\"Hello from generated file\")\n- run ls -la src/agents/generated')

# Programmatic notebook creation example (nbformat)
nb = nbformat.v4.new_notebook()
nb.cells.append(nbformat.v4.new_markdown_cell('# Programmatically created notebook'))
nb.cells.append(nbformat.v4.new_code_cell('print("Hello from programmatic notebook")'))
out_path = Path('notebooks/generated_example.ipynb')
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open('w', encoding='utf-8') as f:
    nbformat.write(nb, f)
print('Wrote', out_path)

# nbclient execute demo (reads and executes the generated notebook)
nb2 = nbformat.read(str(out_path), as_version=4)
client = NotebookClient(nb2, timeout=60, kernel_name='python3')
try:
    client.execute()
    print('Execution succeeded')
except CellExecutionError as e:
    print('Execution failed:', e)

# Inspect outputs
for cell in nb2.cells:
    if 'outputs' in cell and cell.outputs:
        print('Cell type:', cell.cell_type)
        for out in cell.outputs:
            if out.output_type == 'stream':
                print('stream:', out.text)
            elif out.output_type in ('execute_result','display_data'):
                print('data keys:', list(out.data.keys()))
            elif out.output_type == 'error':
                print('error:', out.ename, out.evalue)


Examples:
- scaffold demo_site
- generate filename: example.py
  print("Hello from generated file")
- run ls -la src/agents/generated
Wrote notebooks/generated_example.ipynb


Execution succeeded
Cell type: code
stream: Hello from programmatic notebook

